# 06 - Bronze -> Silver (Nordhaus O2C)

Reads the **mirrored** SAP source out of OneLake as Delta and produces the conformed Silver layer.

Established in `Phase3/BRONZE_ACCESS_FINDINGS.md`:

* the mirrored SQL database is **not** a Spark database - `SHOW DATABASES` does not list it, so
  `spark.table("SAP_S4H_Source...")` is unavailable;
* Fabric requires a **schema-enabled lakehouse as the session default** before a notebook can read
  a mirrored DB at all (`OntologyDataLH` qualifies - it exposes `dbo`);
* therefore Bronze is read by **OneLake abfss path**, as plain Delta. No ODBC, no credentials.

The four traps this notebook exists to handle are called out in the source DDL and are the actual
work of the layer:

1. **DATS dates** are `CHAR(8)`; SAP's empty is `'00000000'`, *not* NULL. A naive cast throws; a
   naive `TRY_CAST` silently misclassifies open vs cleared AR.
2. **Leading zeros** on `VBELN`/`KUNNR`/`MATNR`/`POSNR`. Strip consistently or joins drop rows in
   silence.
3. **BSID + BSAD must be UNIONed** into one AR fact - an item lives in exactly one of them.
4. **On-time is measured against `VBAK.VDATU`** (what the customer asked for), *not* `VBEP.EDATU`
   (what we confirmed back). Which baseline you pick moves OTIF enormously, so this notebook
   computes **both** and keeps them side by side rather than picking one and hiding the choice.

   Measured here: 61.3% on-time against requested, 5.4% against confirmed. In real SAP estates the
   confirmed date is usually pushed *out* past the customer request, which flatters OTIF; in this
   generator `confirmed = credit_release + lead` while `requested = order + lead + 0..9 days`, so
   the confirmed date lands *earlier* and is the harsher yardstick. Same lesson, opposite sign -
   which is exactly why both are reported.


In [ ]:
from pyspark.sql import functions as F, Window as W
import notebookutils

WORKSPACE     = "Nordhaus-O2C-Dev"
SQLDB_ITEM    = "SAP_S4H_Source"
BRONZE_SCHEMA = "sap"
SILVER_PREFIX = "slv_"
CLIENT        = "100"          # SAP MANDT - every application table is client-scoped

ROOT = f"abfss://{WORKSPACE}@onelake.dfs.fabric.microsoft.com"
print("OneLake root:", ROOT)

## 1. Locate Bronze

The item folder name for a Fabric SQL database is not documented stably, so this discovers it
rather than hard-coding it, and **fails loudly** with the actual listing if it cannot find `VBAK`.
A wrong guess here would otherwise surface much later as an empty Silver table.

In [ ]:
def _ls(path):
    """List a OneLake path, returning None instead of raising when it does not exist."""
    try:
        return [f.name.rstrip("/") for f in notebookutils.fs.ls(path)]
    except Exception:
        return None

items = _ls(ROOT)
if items is None:
    raise RuntimeError(f"Cannot list {ROOT} - check the workspace name.")

print("Workspace items in OneLake:")
for i in items:
    print("   ", i)

candidates = [i for i in items if i.lower().startswith(SQLDB_ITEM.lower())]
if not candidates:
    raise RuntimeError(f"No OneLake item starts with {SQLDB_ITEM!r}. Saw: {items}")

BRONZE_ROOT = None
for cand in candidates:
    for sub in ("Tables", f"Tables/{BRONZE_SCHEMA}", f"Tables/dbo"):
        p = f"{ROOT}/{cand}/{sub}"
        listing = _ls(p)
        if not listing:
            continue
        print(f"\n{p}")
        for x in sorted(listing)[:60]:
            print("   -", x)
        if any(x.upper() == "VBAK" for x in listing):
            BRONZE_ROOT = p

if BRONZE_ROOT is None:
    raise RuntimeError("Found the SQL database item but no folder containing VBAK. See listing above.")

print("\nBRONZE_ROOT =", BRONZE_ROOT)

## 2. Conversion helpers

`dats` and `alpha` are the two functions that decide whether this layer is correct. Everything
downstream depends on them being applied *uniformly* - a single table that skips `alpha` silently
loses its joins.

In [ ]:
def dats(col):
    """SAP DATS CHAR(8) -> DateType. '00000000' and '' are SAP's empty, and must become NULL."""
    t = F.trim(F.col(col))
    return F.to_date(F.when(t.isin("", "00000000"), None).otherwise(t), "yyyyMMdd")

def alpha(col):
    """SAP ALPHA output conversion: strip leading zeros from a key ('0000012345' -> '12345')."""
    t = F.trim(F.col(col))
    s = F.regexp_replace(t, "^0+", "")
    return (F.when(t.isNull() | (t == ""), None)
             .otherwise(F.when(s == "", F.lit("0")).otherwise(s)))

def flag(col):
    """SAP single-char indicator -> boolean. '' is false, anything else is true."""
    t = F.trim(F.col(col))
    return F.when(t.isNull() | (t == ""), False).otherwise(True)

def txt(col, alias):
    return F.trim(F.col(col)).alias(alias)

def bronze(name):
    """Read one mirrored SAP table and scope it to the SAP client."""
    df = spark.read.format("delta").load(f"{BRONZE_ROOT}/{name}")
    for c in ("MANDT", "MANDANT", "CLIENT"):
        if c in df.columns:
            return df.filter(F.trim(F.col(c)) == CLIENT)
    return df

In [ ]:
BRONZE_TABLES = [
    "T001","TVKO","TVTW","TSPA","T001W","TVAK","TVAKT","TVAUT",
    "KNA1","KNVV","KNB1","ADRC","MARA","MARC","MVKE","MAKT",
    "VBAK","VBAP","VBEP","VBKD","VBPA","LIKP","LIPS","VBRK","VBRP",
    "PRCD_ELEMENTS","BKPF","BSID","BSAD","VBFA","CDHDR","CDPOS",
]

T = {}
for t in BRONZE_TABLES:
    T[t] = bronze(t).cache()

print(f"{'table':16s}{'rows':>10s}")
bronze_counts = {}
for t in BRONZE_TABLES:
    bronze_counts[t] = T[t].count()
    print(f"{t:16s}{bronze_counts[t]:>10,}")
print(f"\n{'TOTAL':16s}{sum(bronze_counts.values()):>10,}")

## 3. Master data

In [ ]:
adrc = T["ADRC"].select(
    alpha("ADDRNUMBER").alias("AddressId"),
    txt("STREET", "Street"),
    txt("POST_CODE1", "PostalCode"),
)

slv_customer = (T["KNA1"]
    .select(
        alpha("KUNNR").alias("CustomerId"),
        txt("NAME1", "CustomerName"),
        txt("LAND1", "CountryKey"),
        txt("ORT01", "City"),
        txt("REGIO", "Region"),
        txt("PSTLZ", "PostalCode"),
        txt("KTOKD", "AccountGroup"),
        txt("BRSCH", "IndustryKey"),
        alpha("ADRNR").alias("AddressId"),
        dats("ERDAT").alias("CreatedDate"),
        flag("LOEVM").alias("IsDeleted"))
    .join(adrc.drop("PostalCode"), "AddressId", "left"))

slv_customer_sales_area = (T["KNVV"]
    .select(
        alpha("KUNNR").alias("CustomerId"),
        txt("VKORG", "SalesOrg"),
        txt("VTWEG", "DistributionChannel"),
        txt("SPART", "Division"),
        txt("KDGRP", "CustomerGroup"),
        txt("BZIRK", "SalesDistrict"),
        txt("KONDA", "PriceGroup"),
        txt("INCO1", "Incoterms"),
        txt("ZTERM", "PaymentTerms"),
        txt("VSBED", "ShippingConditions"),
        flag("LOEVM").alias("IsDeleted")))

makt_en = T["MAKT"].filter(F.trim(F.col("SPRAS")) == "E").select(
    alpha("MATNR").alias("MaterialId"), txt("MAKTX", "MaterialName"))

slv_material = (T["MARA"]
    .select(
        alpha("MATNR").alias("MaterialId"),
        txt("MTART", "MaterialType"),
        txt("MATKL", "MaterialGroup"),
        txt("MEINS", "BaseUnit"),
        F.col("BRGEW").alias("GrossWeight"),
        F.col("NTGEW").alias("NetWeight"),
        txt("GEWEI", "WeightUnit"),
        txt("PRDHA", "ProductHierarchy"),
        dats("ERSDA").alias("CreatedDate"),
        flag("LVORM").alias("IsDeleted"))
    .join(makt_en, "MaterialId", "left"))

slv_material_plant = (T["MARC"]
    .select(
        alpha("MATNR").alias("MaterialId"),
        txt("WERKS", "Plant"),
        txt("DISMM", "MrpType"),
        txt("BESKZ", "ProcurementType"),
        F.col("PLIFZ").alias("PlannedDeliveryTimeDays"),
        txt("STRGR", "PlanningStrategy"))
    .withColumn("MakeToOrder", F.col("PlanningStrategy").isin("20", "50")))

slv_plant = T["T001W"].select(
    txt("WERKS", "Plant"), txt("NAME1", "PlantName"),
    txt("LAND1", "CountryKey"), txt("ORT01", "City"), txt("VKORG", "SalesOrg"))

slv_sales_org = T["TVKO"].select(
    txt("VKORG", "SalesOrg"), txt("BUKRS", "CompanyCode"),
    txt("VKOOR", "SalesOrgName"), txt("WAERS", "Currency"))

slv_division = T["TSPA"].select(txt("SPART", "Division"), txt("VTEXT", "DivisionName"))
slv_channel  = T["TVTW"].select(txt("VTWEG", "DistributionChannel"), txt("VTEXT", "ChannelName"))
slv_company  = T["T001"].select(
    txt("BUKRS", "CompanyCode"), txt("BUTXT", "CompanyName"),
    txt("LAND1", "CountryKey"), txt("WAERS", "LocalCurrency"))
slv_rejection_reason = (T["TVAUT"].filter(F.trim(F.col("SPRAS")) == "E")
    .select(txt("ABGRU", "RejectionReason"), txt("BEZEI", "RejectionReasonText")))
slv_order_type = (T["TVAKT"].filter(F.trim(F.col("SPRAS")) == "E")
    .select(txt("AUART", "OrderType"), txt("BEZEI", "OrderTypeText")))

print("master data frames defined")

## 4. Sales orders

`VBEP` carries one row per schedule line; the **first** schedule line holds the confirmed date, so
it is reduced with a window rather than joined blind (a blind join would fan the item grain out).

Ship-to comes from `VBPA` partner function `WE` at header level - roughly 15% of orders ship
somewhere other than the sold-to, which is exactly why one order does not mean one customer.

In [ ]:
vbak, vbap, vbep, vbkd, vbpa = T["VBAK"], T["VBAP"], T["VBEP"], T["VBKD"], T["VBPA"]

order_hdr = vbak.select(
    alpha("VBELN").alias("SalesOrderId"),
    dats("ERDAT").alias("OrderCreatedDate"),
    dats("AUDAT").alias("OrderDocumentDate"),
    txt("AUART", "OrderType"),
    txt("VBTYP", "DocCategory"),
    txt("VKORG", "SalesOrg"),
    txt("VTWEG", "DistributionChannel"),
    txt("SPART", "Division"),
    alpha("KUNNR").alias("SoldToId"),
    F.col("NETWR").alias("OrderNetValue"),
    txt("WAERK", "DocCurrency"),
    dats("VDATU").alias("RequestedDeliveryDate"),
    flag("LIFSK").alias("HasDeliveryBlock"),
    flag("FAKSK").alias("HasBillingBlock"),
    txt("CMGST", "CreditStatus"),
    txt("GBSTK", "OverallStatus"),
    txt("LFSTK", "DeliveryStatus"),
    txt("FKSTK", "BillingStatus"),
)

_w = W.partitionBy("VBELN", "POSNR").orderBy(F.col("ETENR").asc())
schedule_first = (vbep
    .withColumn("_rn", F.row_number().over(_w))
    .filter(F.col("_rn") == 1)
    .select(
        alpha("VBELN").alias("SalesOrderId"),
        alpha("POSNR").alias("SalesOrderItem"),
        dats("EDATU").alias("ConfirmedDeliveryDate"),
        F.col("BMENG").alias("ConfirmedQty")))

order_terms = (vbkd.filter(F.trim(F.col("POSNR")) == "000000")
    .select(
        alpha("VBELN").alias("SalesOrderId"),
        txt("ZTERM", "PaymentTerms"),
        txt("INCO1", "Incoterms"),
        F.col("KURSK").alias("ExchangeRate")))

ship_to = (vbpa
    .filter((F.trim(F.col("PARVW")) == "WE") & (F.trim(F.col("POSNR")) == "000000"))
    .select(alpha("VBELN").alias("SalesOrderId"), alpha("KUNNR").alias("ShipToId"))
    .dropDuplicates(["SalesOrderId"]))

order_itm = vbap.select(
    alpha("VBELN").alias("SalesOrderId"),
    alpha("POSNR").alias("SalesOrderItem"),
    alpha("MATNR").alias("MaterialId"),
    txt("ARKTX", "ItemText"),
    txt("PSTYV", "ItemCategory"),
    txt("WERKS", "Plant"),
    F.col("KWMENG").alias("OrderQty"),
    txt("VRKME", "SalesUnit"),
    F.col("NETWR").alias("ItemNetValue"),
    F.col("NETPR").alias("NetPrice"),
    F.col("KZWI1").alias("ListValue"),
    F.col("WAVWR").alias("ItemCost"),
    txt("MATKL", "MaterialGroup"),
    txt("ABGRU", "RejectionReason"),
)

slv_sales_order_item = (order_itm
    .join(order_hdr, "SalesOrderId", "left")
    .join(schedule_first, ["SalesOrderId", "SalesOrderItem"], "left")
    .join(order_terms, "SalesOrderId", "left")
    .join(ship_to, "SalesOrderId", "left")
    .withColumn("IsRejected", F.coalesce(F.col("RejectionReason"), F.lit("")) != "")
    .withColumn("ShipToId", F.coalesce(F.col("ShipToId"), F.col("SoldToId")))
    .withColumn("ShipsElsewhere", F.col("ShipToId") != F.col("SoldToId"))
    .withColumn("DiscountAmount", F.col("ListValue") - F.col("ItemNetValue"))
    .withColumn("DiscountPct", F.when(F.col("ListValue") > 0,
                                      (F.col("ListValue") - F.col("ItemNetValue")) / F.col("ListValue"))))

print("sales order items:", slv_sales_order_item.count())

## 5. Deliveries and billing

In [ ]:
likp, lips, vbrk, vbrp = T["LIKP"], T["LIPS"], T["VBRK"], T["VBRP"]

delivery_hdr = likp.select(
    alpha("VBELN").alias("DeliveryId"),
    dats("ERDAT").alias("DeliveryCreatedDate"),
    txt("LFART", "DeliveryType"),
    txt("VKORG", "SalesOrg"),
    alpha("KUNNR").alias("ShipToId"),
    alpha("KUNAG").alias("SoldToId"),
    dats("LFDAT").alias("PlannedDeliveryDate"),
    dats("WADAT").alias("PlannedGoodsIssueDate"),
    dats("WADAT_IST").alias("ActualGoodsIssueDate"),
    dats("KODAT").alias("PickingDate"),
    F.col("BTGEW").alias("TotalWeight"),
    txt("VSBED", "ShippingConditions"),
    txt("ROUTE", "Route"),
    txt("WBSTK", "GoodsMovementStatus"),
)

slv_delivery_item = (lips
    .select(
        alpha("VBELN").alias("DeliveryId"),
        alpha("POSNR").alias("DeliveryItem"),
        alpha("MATNR").alias("MaterialId"),
        txt("WERKS", "Plant"),
        F.col("LFIMG").alias("DeliveredQty"),
        txt("VRKME", "SalesUnit"),
        alpha("VGBEL").alias("SalesOrderId"),
        alpha("VGPOS").alias("SalesOrderItem"),
        F.col("NETWR").alias("DeliveryNetValue"))
    .join(delivery_hdr, "DeliveryId", "left"))

billing_hdr = vbrk.select(
    alpha("VBELN").alias("BillingDocId"),
    txt("FKART", "BillingType"),
    txt("VBTYP", "DocCategory"),
    dats("FKDAT").alias("BillingDate"),
    txt("BUKRS", "CompanyCode"),
    txt("VKORG", "SalesOrg"),
    alpha("KUNRG").alias("PayerId"),
    alpha("KUNAG").alias("SoldToId"),
    F.col("NETWR").alias("BillingNetValue"),
    F.col("MWSBK").alias("TaxAmount"),
    txt("WAERK", "DocCurrency"),
    F.col("KURRF").alias("ExchangeRate"),
    txt("ZTERM", "PaymentTerms"),
    flag("FKSTO").alias("IsCancelled"),
    alpha("BELNR").alias("AccountingDocId"),
)

slv_billing_item = (vbrp
    .select(
        alpha("VBELN").alias("BillingDocId"),
        alpha("POSNR").alias("BillingItem"),
        alpha("MATNR").alias("MaterialId"),
        F.col("FKIMG").alias("BilledQty"),
        F.col("NETWR").alias("BillingItemNetValue"),
        F.col("WAVWR").alias("BillingItemCost"),
        F.col("KZWI1").alias("BillingItemListValue"),
        txt("MATKL", "MaterialGroup"),
        alpha("AUBEL").alias("SalesOrderId"),
        alpha("AUPOS").alias("SalesOrderItem"),
        alpha("VGBEL").alias("DeliveryId"),
        alpha("VGPOS").alias("DeliveryItem"),
        txt("KNUMV", "ConditionRecord"))
    .join(billing_hdr, "BillingDocId", "left")
    .withColumn("IsCreditMemo", F.col("BillingType").isin("G2", "RE")))

print("delivery items:", slv_delivery_item.count(), "| billing items:", slv_billing_item.count())

## 6. Accounts receivable - the BSID U BSAD union

SAP splits AR across two identically-shaped tables: `BSID` holds **open** items, `BSAD` holds
**cleared** ones, and a given item is in exactly one of them at any moment. Querying either alone
produces a confidently wrong DSO. `SourceTable` is kept so provenance survives into Gold.

In [ ]:
AR_COLS = ["BUKRS","KUNNR","AUGDT","AUGBL","GJAHR","BELNR","BUZEI","BUDAT","BLDAT",
           "WAERS","XBLNR","BLART","SHKZG","DMBTR","WRBTR","ZFBDT","ZBD1T","ZTERM"]

ar_union = (T["BSID"].select(*AR_COLS).withColumn("SourceTable", F.lit("BSID"))
    .unionByName(T["BSAD"].select(*AR_COLS).withColumn("SourceTable", F.lit("BSAD"))))

slv_ar_item = (ar_union
    .select(
        txt("BUKRS", "CompanyCode"),
        alpha("KUNNR").alias("CustomerId"),
        alpha("BELNR").alias("AccountingDocId"),
        txt("GJAHR", "FiscalYear"),
        txt("BUZEI", "LineItem"),
        dats("BUDAT").alias("PostingDate"),
        dats("BLDAT").alias("DocumentDate"),
        dats("ZFBDT").alias("BaselineDate"),
        dats("AUGDT").alias("ClearingDate"),
        alpha("AUGBL").alias("ClearingDocId"),
        txt("WAERS", "Currency"),
        alpha("XBLNR").alias("BillingDocRef"),
        txt("BLART", "DocumentType"),
        txt("SHKZG", "DebitCreditInd"),
        F.col("DMBTR").alias("AmountLocal"),
        F.col("WRBTR").alias("AmountDoc"),
        F.col("ZBD1T").alias("CashDiscountDays"),
        txt("ZTERM", "PaymentTerms"),
        F.col("SourceTable"))
    .withColumn("IsOpen", F.col("ClearingDate").isNull())
    .withColumn("DueDate", F.expr("date_add(BaselineDate, cast(coalesce(CashDiscountDays, 0) as int))"))
    .withColumn("DaysToPay", F.datediff("ClearingDate", "BaselineDate"))
    .withColumn("DaysOverdue",
                F.when(F.col("ClearingDate").isNull(), F.datediff(F.current_date(), F.col("DueDate")))
                 .otherwise(F.datediff("ClearingDate", "DueDate")))
    .withColumn("IsOverdue", F.col("DaysOverdue") > 0)
    .withColumn("SignedAmountLocal",
                F.when(F.col("DebitCreditInd") == "H", -F.col("AmountLocal")).otherwise(F.col("AmountLocal"))))

print("AR items:", slv_ar_item.count(),
      "| open:", slv_ar_item.filter("IsOpen").count(),
      "| cleared:", slv_ar_item.filter("not IsOpen").count())

## 7. Document flow

`VBFA` is the edge list of the O2C graph and is what the ontology will traverse in Phase 6. It is
kept as edges rather than flattened, because the relationships are genuinely many-to-many:
deliveries consolidate order items, and the weekly billing run consolidates deliveries.

In [ ]:
CATEGORY = {"C": "SalesOrder", "J": "Delivery", "M": "Invoice",
            "O": "CreditMemo", "H": "Return", "T": "Return"}
_cat_map = F.create_map([F.lit(x) for kv in CATEGORY.items() for x in kv])

slv_doc_flow = (T["VBFA"]
    .select(
        alpha("VBELV").alias("PrecedingDocId"),
        alpha("POSNV").alias("PrecedingItem"),
        alpha("VBELN").alias("SubsequentDocId"),
        alpha("POSNN").alias("SubsequentItem"),
        txt("VBTYP_V", "PrecedingCategory"),
        txt("VBTYP_N", "SubsequentCategory"),
        F.col("RFMNG").alias("ReferencedQty"),
        F.col("RFWRT").alias("ReferencedValue"),
        dats("ERDAT").alias("FlowCreatedDate"))
    .withColumn("PrecedingType",  _cat_map[F.col("PrecedingCategory")])
    .withColumn("SubsequentType", _cat_map[F.col("SubsequentCategory")]))

print("document flow edges:", slv_doc_flow.count())
slv_doc_flow.groupBy("PrecedingType", "SubsequentType").count().orderBy(F.desc("count")).show(20, False)

## 8. Order fulfilment - OTIF at order-item grain

One order item can span several deliveries, so delivered quantity is **aggregated to the item**
before being compared with the ordered quantity. Comparing a single delivery line against the
order line is the classic way in-full rates come out wrong.

`IsOnTime` uses the customer's requested date. `IsOnTimeVsConfirmed` uses the internally confirmed
date and exists purely so the report can show the gap between the two - the flattering number and
the honest one, side by side.

In [ ]:
delivered = (slv_delivery_item
    .filter(F.col("SalesOrderId").isNotNull())
    .groupBy("SalesOrderId", "SalesOrderItem")
    .agg(F.sum("DeliveredQty").alias("DeliveredQty"),
         F.countDistinct("DeliveryId").alias("DeliveryCount"),
         F.min("ActualGoodsIssueDate").alias("FirstGoodsIssueDate"),
         F.max("ActualGoodsIssueDate").alias("LastGoodsIssueDate")))

billed = (slv_billing_item
    .filter(F.col("SalesOrderId").isNotNull())
    .groupBy("SalesOrderId", "SalesOrderItem")
    .agg(F.sum("BilledQty").alias("BilledQty"),
         F.sum("BillingItemNetValue").alias("BilledNetValue"),
         F.countDistinct("BillingDocId").alias("BillingDocCount"),
         F.min("BillingDate").alias("FirstBillingDate"),
         F.max("BillingDate").alias("LastBillingDate")))

slv_order_fulfilment = (slv_sales_order_item
    .join(delivered, ["SalesOrderId", "SalesOrderItem"], "left")
    .join(billed,    ["SalesOrderId", "SalesOrderItem"], "left")
    .withColumn("DeliveredQty", F.coalesce(F.col("DeliveredQty"), F.lit(0.0)))
    .withColumn("BilledQty",    F.coalesce(F.col("BilledQty"), F.lit(0.0)))
    .withColumn("DeliveryCount", F.coalesce(F.col("DeliveryCount"), F.lit(0)))
    .withColumn("IsDelivered", F.col("DeliveredQty") > 0)
    .withColumn("IsInFull", F.col("DeliveredQty") >= F.col("OrderQty"))
    .withColumn("IsOnTime",
                F.col("LastGoodsIssueDate").isNotNull()
                & (F.col("LastGoodsIssueDate") <= F.col("RequestedDeliveryDate")))
    .withColumn("IsOnTimeVsConfirmed",
                F.col("LastGoodsIssueDate").isNotNull()
                & (F.col("LastGoodsIssueDate") <= F.col("ConfirmedDeliveryDate")))
    .withColumn("IsOTIF", F.col("IsInFull") & F.col("IsOnTime"))
    .withColumn("IsSplitDelivery", F.col("DeliveryCount") > 1)
    .withColumn("IsOpenBacklog", (~F.col("IsRejected")) & (F.col("DeliveredQty") < F.col("OrderQty")))
    .withColumn("DeliveryDelayDays", F.datediff("LastGoodsIssueDate", "RequestedDeliveryDate"))
    .withColumn("OrderToGoodsIssueDays", F.datediff("LastGoodsIssueDate", "OrderCreatedDate"))
    .withColumn("GoodsIssueToInvoiceDays", F.datediff("LastBillingDate", "LastGoodsIssueDate"))
    .withColumn("OrderToInvoiceDays", F.datediff("LastBillingDate", "OrderCreatedDate")))

_scope = slv_order_fulfilment.filter("not IsRejected")
_n = _scope.count()
print(f"OTIF scope (non-rejected items): {_n:,}")
if _n:
    _agg = _scope.agg(
        F.avg(F.col("IsInFull").cast("double")).alias("in_full"),
        F.avg(F.col("IsOnTime").cast("double")).alias("on_time_requested"),
        F.avg(F.col("IsOnTimeVsConfirmed").cast("double")).alias("on_time_confirmed"),
        F.avg(F.col("IsOTIF").cast("double")).alias("otif")).collect()[0]
    print(f"  in-full                : {_agg['in_full']:.1%}")
    print(f"  on-time vs REQUESTED   : {_agg['on_time_requested']:.1%}   <- what the customer asked for")
    print(f"  on-time vs CONFIRMED   : {_agg['on_time_confirmed']:.1%}   <- what we committed back")
    print(f"  OTIF                   : {_agg['otif']:.1%}")

## 9. Order change history

In [ ]:
slv_order_change = (T["CDPOS"]
    .join(T["CDHDR"], ["OBJECTCLAS", "OBJECTID", "CHANGENR"], "inner")
    .select(
        alpha("OBJECTID").alias("SalesOrderId"),
        txt("CHANGENR", "ChangeNumber"),
        dats("UDATE").alias("ChangeDate"),
        txt("USERNAME", "ChangedBy"),
        txt("TCODE", "TransactionCode"),
        txt("TABNAME", "ChangedTable"),
        txt("FNAME", "ChangedField"),
        txt("CHNGIND", "ChangeIndicator"),
        txt("VALUE_OLD", "OldValue"),
        txt("VALUE_NEW", "NewValue")))

print("order change rows:", slv_order_change.count())
slv_order_change.groupBy("ChangedField").count().orderBy(F.desc("count")).show(10, False)

## 10. Write Silver

Delta, `overwrite`, into the default (schema-enabled) lakehouse. Silver keeps `Decimal` - the
`Decimal -> Double` conversion is a **Gold** requirement, because Fabric Graph returns null for
every Decimal property. Doing it here would be premature and would lose precision for no reason.

In [ ]:
SILVER = {
    "customer":            slv_customer,
    "customer_sales_area": slv_customer_sales_area,
    "material":            slv_material,
    "material_plant":      slv_material_plant,
    "plant":               slv_plant,
    "sales_org":           slv_sales_org,
    "division":            slv_division,
    "distribution_channel": slv_channel,
    "company":             slv_company,
    "rejection_reason":    slv_rejection_reason,
    "order_type":          slv_order_type,
    "sales_order_item":    slv_sales_order_item,
    "delivery_item":       slv_delivery_item,
    "billing_item":        slv_billing_item,
    "ar_item":             slv_ar_item,
    "doc_flow":            slv_doc_flow,
    "order_fulfilment":    slv_order_fulfilment,
    "order_change":        slv_order_change,
}

written = {}
for name, df in SILVER.items():
    table = f"{SILVER_PREFIX}{name}"
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true").saveAsTable(table))
    written[table] = spark.table(table).count()
    print(f"{table:30s}{written[table]:>10,}")

## 11. Validation gate

These assertions inspect the **written tables**, not the in-memory DataFrames. A check that can
pass while the artefact is missing is the wrong check - that lesson cost a full silent run in
Phase 1.

In [ ]:
failures = []

def check(label, ok, detail=""):
    print(f"{'PASS' if ok else 'FAIL'}  {label}" + (f"  - {detail}" if detail else ""))
    if not ok:
        failures.append(label)

for table, n in written.items():
    check(f"{table} is non-empty", n > 0, f"{n:,} rows")

soi_n = spark.table(f"{SILVER_PREFIX}sales_order_item").count()
check("sales_order_item preserves VBAP grain", soi_n == bronze_counts["VBAP"],
      f"silver {soi_n:,} vs VBAP {bronze_counts['VBAP']:,}")

ful_n = spark.table(f"{SILVER_PREFIX}order_fulfilment").count()
check("order_fulfilment preserves order-item grain", ful_n == bronze_counts["VBAP"],
      f"{ful_n:,} vs {bronze_counts['VBAP']:,}")

ar_n = spark.table(f"{SILVER_PREFIX}ar_item").count()
check("ar_item == BSID + BSAD", ar_n == bronze_counts["BSID"] + bronze_counts["BSAD"],
      f"{ar_n:,} vs {bronze_counts['BSID']:,}+{bronze_counts['BSAD']:,}")

ar = spark.table(f"{SILVER_PREFIX}ar_item")
check("every BSID row is open",  ar.filter("SourceTable='BSID' and not IsOpen").count() == 0)
check("every BSAD row is cleared", ar.filter("SourceTable='BSAD' and IsOpen").count() == 0)

check("no DATS sentinel leaked through as a real date",
      ar.filter("ClearingDate < date'1900-01-01' or PostingDate < date'1900-01-01'").count() == 0)

flow = spark.table(f"{SILVER_PREFIX}doc_flow")
check("all document-flow categories are mapped",
      flow.filter("SubsequentType is null").count() == 0)

ful = spark.table(f"{SILVER_PREFIX}order_fulfilment")
check("OTIF implies in-full and on-time",
      ful.filter("IsOTIF and (not IsInFull or not IsOnTime)").count() == 0)
check("no negative delivered quantity", ful.filter("DeliveredQty < 0").count() == 0)
check("rejected items were never delivered",
      ful.filter("IsRejected and DeliveredQty > 0").count() == 0)

# --- many-to-many document flow -------------------------------------------------
# The ontology is only interesting if the document flow is a genuine graph, not a tree.
# Measured 2026-08-05 against this dataset:
#     deliveries total                       14,268
#     deliveries carrying >1 ORDER ITEM       1,371
#     deliveries spanning  >1 ORDER               0   <- zero BY DESIGN
#     invoices consolidating >1 delivery      2,640
#     order items split across >1 delivery    1,838
# The generator consolidates shipments by goods-issue date WITHIN an order, so a delivery
# never spans two orders. An earlier version of this check tested for multi-ORDER deliveries
# and failed for that reason: the test was wrong, not the data. Do not change it back.
dli = spark.table(f"{SILVER_PREFIX}delivery_item")
bli = spark.table(f"{SILVER_PREFIX}billing_item")

check("deliveries consolidate multiple order ITEMS",
      dli.filter(F.col("SalesOrderId").isNotNull())
         .groupBy("DeliveryId")
         .agg(F.countDistinct(F.concat_ws("|", "SalesOrderId", "SalesOrderItem")).alias("n"))
         .filter("n > 1").count() > 0)

check("invoices consolidate multiple deliveries",
      bli.filter(F.col("DeliveryId").isNotNull())
         .groupBy("BillingDocId").agg(F.countDistinct("DeliveryId").alias("n"))
         .filter("n > 1").count() > 0)

check("order items split across multiple deliveries",
      dli.filter(F.col("SalesOrderId").isNotNull())
         .groupBy("SalesOrderId", "SalesOrderItem")
         .agg(F.countDistinct("DeliveryId").alias("n"))
         .filter("n > 1").count() > 0)

check("deliveries never span two sales orders (generator design, not a defect)",
      dli.filter(F.col("SalesOrderId").isNotNull())
         .groupBy("DeliveryId").agg(F.countDistinct("SalesOrderId").alias("n"))
         .filter("n > 1").count() == 0)

check("leading zeros stripped consistently",
      spark.table(f"{SILVER_PREFIX}sales_order_item")
           .filter("SalesOrderId rlike '^0'").count() == 0)

print()
if failures:
    raise AssertionError(f"{len(failures)} Silver validation check(s) failed: {failures}")
print(f"All {len(written)} Silver tables written and validated.")